In [69]:
import importlib
import performance_shr
importlib.reload(performance_shr)

<module 'performance_shr' from 'd:\\Astro\\Bin\\alpaca-benro-polaris\\performance\\performance_shr.py'>

In [70]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from performance_shr import convert_to_float, convert_to_bool, convert_to_arcsec, plot_formating

In [83]:
csv_filename = './fits_extract.csv'

fits_converters = {
    'filename':             str,
    'status':               str,
    'date_obs':             str,
    'object':               str,
    'filter':               str,
    'exptime_s':            convert_to_float,
    'site_lat':             convert_to_float,
    'site_lon':             convert_to_float,
    'p_ra':                 convert_to_float,
    'p_dec':                convert_to_float,
    'p_az':                 convert_to_float,
    'p_alt':                convert_to_float,
    'p_roll':               convert_to_float,
    'theta1':               convert_to_float,
    'theta2':               convert_to_float,
    'theta3':               convert_to_float,
    'solved_ra':            convert_to_float,
    'solved_dec':           convert_to_float,
    'solved_pa':            convert_to_float,
    'solved_az':            convert_to_float,
    'solved_alt':           convert_to_float,
    'solved_roll':          convert_to_float,
    'dev_az_arcmin':        convert_to_float,
    'dev_alt_arcmin':       convert_to_float,
    'dev_roll_arcmin':      convert_to_float,
    'crota2':               convert_to_float,
    'cd1_1':                convert_to_float,
    'cd1_2':                convert_to_float,
    'cd2_1':                convert_to_float,
    'cd2_2':                convert_to_float,
    'pa_source':            str,
    'parallactic_angle':    convert_to_float,
    'pixel_scale_arcsec':   convert_to_float,
}

d = pd.read_csv(csv_filename, converters=fits_converters)
d['date_obs'] = pd.to_datetime(d['date_obs'], utc=True)

alt_bins   = [0,    32.5, 42.5, 50,  57.5, 65,  90]
alt_labels = ['~25°', '~40°', '~45°', '~55°', '~60°', '~70°']

az_bins    = [0,   110,  170,  210,  270,  320,  360]
az_labels  = ['~70°', '~150°', '~190°', '~230°', '~309°', '~330°']

# Derived bin fields
d['alt_bin'] = pd.cut(d['p_alt'], bins=alt_bins, labels=alt_labels)
d['az_bin']  = pd.cut(d['p_az'],  bins=az_bins,  labels=az_labels)
d['bin']     = 'alt' + d['alt_bin'].astype(str) + ' / az' + d['az_bin'].astype(str)

# Convenience subsets
solved   = d[d['status'] == 'solved'].copy()
unsolved = d[d['status'] == 'unsolved'].copy()

# Derived solved fields
# Component 3 removed — should show no p_roll dependence, but still has C1+C2
solved['roll_error_predicted'] = (0.9742 * np.tan(np.radians(solved['p_alt'])) + 0.2509) * solved['p_roll']
solved['dev_roll_fixed'] = solved['dev_roll_arcmin'] - solved['roll_error_predicted']

# Convenience subsets without small bins
plot_df = solved[
    ~solved['alt_bin'].isin(['~45°', '~60°']) &
    ~solved['az_bin'].isin(['~190°', '~309°'])
]

print(f"Loaded {len(d)} rows — {len(solved)} solved, {len(unsolved)} unsolved")
solved.columns

Loaded 1162 rows — 893 solved, 269 unsolved


Index(['filename', 'status', 'date_obs', 'object', 'filter', 'exptime_s',
       'site_lat', 'site_lon', 'p_ra', 'p_dec', 'p_az', 'p_alt', 'p_roll',
       'theta1', 'theta2', 'theta3', 'solved_ra', 'solved_dec', 'solved_pa',
       'solved_az', 'solved_alt', 'solved_roll', 'dev_az_arcmin',
       'dev_alt_arcmin', 'dev_roll_arcmin', 'crota2', 'cd1_1', 'cd1_2',
       'cd2_1', 'cd2_2', 'pa_source', 'parallactic_angle',
       'pixel_scale_arcsec', 'alt_bin', 'az_bin', 'bin',
       'roll_error_predicted', 'dev_roll_fixed'],
      dtype='object')

In [72]:
def plot_deviations(x_field, df=solved):
    fig = go.Figure()

    for y_field, name in [
        ('dev_az_arcmin',   'Az dev'),
        ('dev_alt_arcmin',  'Alt dev'),
        ('dev_roll_arcmin', 'Roll dev'),
    ]:
        fig.add_trace(go.Scatter(
            x=df[x_field], y=df[y_field],
            mode='markers', name=name,
            marker=dict(size=6)
        ))

    fig.update_layout(
        title=dict(text=f'Deviations vs {x_field}', x=0.5, font=dict(size=24, family='Arial')),
        xaxis_title=x_field,
        yaxis_title='Deviation (arcmin)',
        plot_bgcolor='rgba(200, 200, 250, 0.5)',
        legend=dict(x=0.01, y=0.99),
        hovermode='x unified'
    )

    fig.show()

In [73]:
plot_deviations('p_roll')
plot_deviations('p_alt')
plot_deviations('p_az')
plot_deviations('theta1')
plot_deviations('theta2')
plot_deviations('theta3')
plot_deviations('date_obs')


In [88]:


fig = px.scatter(
    plot_df,
    x='dev_roll_arcmin',
    y='dev_roll_fixed',
    color='p_roll',
    facet_col='alt_bin',
    facet_row='az_bin',
    color_continuous_scale='Viridis',
    labels={
        'dev_roll_arcmin': 'Roll dev (arcmin)',
        'dev_roll_fixed':  'Roll dev fixed (arcmin)',
        'p_roll':          'p_roll (°)',
    },
    title='Dev Roll Fixed vs Dev Roll — Columns by Alt, Rows by Az, ',
    category_orders={
        'alt_bin': alt_labels_filtered,
        'az_bin':  az_labels_filtered,
    },
)

# Add a reference line y=x (perfect 1:1 relationship)
for annotation in fig.layout.annotations:
    pass

x_range = [plot_df['dev_roll_arcmin'].min(), plot_df['dev_roll_arcmin'].max()]

fig.add_shape(
    type='line', line=dict(dash='dash', color='red', width=1),
    x0=x_range[0], x1=x_range[1],
    y0=x_range[0], y1=x_range[1],
    xref='x', yref='y'
)

fig.update_layout(
    title=dict(x=0.5, font=dict(size=24, family='Arial')),
    plot_bgcolor='rgba(200, 200, 250, 0.5)',
    height=1000,
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))

fig.show()

In [76]:
fig = px.scatter(
    plot_df,
    x='dev_roll_arcmin',
    y='dev_az_arcmin',
    color='p_roll',
    symbol='alt_bin',
    facet_col='az_bin',
    color_continuous_scale='Viridis',
    labels={
        'dev_roll_arcmin': 'Roll dev (arcmin)',
        'dev_az_arcmin':    'Az dev',
        'p_roll':          'p_roll (°)',
        'alt_bin':         'Alt bin',
    },
    title='Az dev vs Roll dev — faceted by Az, symbol by Alt',
    category_orders={
        'alt_bin': alt_labels_filtered,
        'az_bin':  az_labels_filtered,
    },
)

x_range = [plot_df['dev_roll_arcmin'].min(), plot_df['dev_roll_arcmin'].max()]
fig.add_shape(
    type='line', line=dict(dash='dash', color='red', width=1),
    x0=x_range[0], x1=x_range[1],
    y0=x_range[0], y1=x_range[1],
    xref='x', yref='y'
)

fig.update_layout(
    title=dict(x=0.5, font=dict(size=24, family='Arial')),
    plot_bgcolor='rgba(200, 200, 250, 0.5)',
    height=900,
    coloraxis_colorbar=dict(
        x=1.025,  # move colorbar further right
        len=0.6  # optional: shrink vertically
    )
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))

fig.show()

In [77]:
fig = px.scatter(
    plot_df,
    x='theta3',
    y='p_roll',
    color='theta2',
    facet_col='alt_bin',
    facet_row='az_bin',
    color_continuous_scale='Viridis',
    labels={
        'theta3': 'Theta3 (°)',
        'theta2':     'Theta2 (°)',
        'p_roll':          'p_roll (°)',
    },
    title='Roll vs Theta3 — Columns by Alt, Rows by Az, Color by Theta2 ',
    category_orders={
        'alt_bin': alt_labels_filtered,
        'az_bin':  az_labels_filtered,
    },
)


fig.update_layout(
    title=dict(x=0.5, font=dict(size=24, family='Arial')),
    plot_bgcolor='rgba(200, 200, 250, 0.5)',
    height=1000,
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))

fig.show()

In [78]:
# ─────────────────────────────────────────────────────────────────────────────
# ROLL ERROR MODEL FIT
# Decomposes dev_roll into three components:
#   1. Global alignment offset (SPA-only bias — absorbed by QUEST in production)
#   2. Az-dependent offset     (polar misalignment — also absorbed by QUEST)
#   3. Roll-dependent residual (what QUEST cannot fix — the target of this model)
#
# Final model:
#   roll_error(arcmin) = (roll_model_a · tan(alt) + roll_model_b) · p_roll
# ─────────────────────────────────────────────────────────────────────────────

from scipy.optimize import curve_fit
from scipy import stats

MODEL_ALT_BINS = ['~25°', '~40°', '~55°', '~70°']
MODEL_AZ_BINS  = ['~70°', '~150°', '~230°', '~330°']

model_df = solved[
    solved['alt_bin'].isin(MODEL_ALT_BINS) &
    solved['az_bin'].isin(MODEL_AZ_BINS)
].copy()

# ── Component 1: Global SPA bias ──────────────────────────────────────────────

_global_mean = model_df['dev_roll_arcmin'].mean()
_global_std  = model_df['dev_roll_arcmin'].std()
_global_n    = len(model_df)

print("── Component 1: Global alignment offset (SPA bias) ─────────")
print(f"   Mean dev_roll        : {_global_mean:+.1f} arcmin  ({_global_mean/60:+.3f}°)")
print(f"   Std  dev_roll        : {_global_std:.1f} arcmin")
print(f"   N (solved, 4×4 grid) : {_global_n}")
print(f"   Interpretation       : mount has ~{_global_mean/60:.2f}° global roll bias under SPA-only")
print(f"                          QUEST will absorb this with any sync point")

# ── Component 2: Az-dependent offset (polar misalignment) ─────────────────────

def sinusoidal_az(az_deg, amplitude, az0_deg, offset):
    return amplitude * np.cos(np.radians(az_deg - az0_deg)) + offset

_az_bin_centres = {'~70°': 70, '~150°': 150, '~230°': 230, '~330°': 330}
_az_dev_roll_means = (
    model_df
    .groupby('az_bin', observed=True)['dev_roll_arcmin']
    .mean()
    .reindex(MODEL_AZ_BINS)
)
_az_xs = np.array([_az_bin_centres[k] for k in _az_dev_roll_means.index])
_az_ys = _az_dev_roll_means.values

_popt_az, _ = curve_fit(sinusoidal_az, _az_xs, _az_ys, p0=[60, 90, 150])
roll_az_amplitude, roll_az_az0, roll_az_offset = _popt_az

# R² of the Az sinusoid fit
_az_pred   = sinusoidal_az(_az_xs, *_popt_az)
_az_r2     = 1 - np.sum((_az_ys - _az_pred)**2) / np.sum((_az_ys - _az_ys.mean())**2)

# Sinusoid fit to dev_alt as well (independent confirmation of polar tilt)
_az_dev_alt_means = (
    model_df
    .groupby('az_bin', observed=True)['dev_alt_arcmin']
    .mean()
    .reindex(MODEL_AZ_BINS)
    .values
)
_popt_alt, _ = curve_fit(sinusoidal_az, _az_xs, _az_dev_alt_means, p0=[60, 0, 30])
_alt_pred    = sinusoidal_az(_az_xs, *_popt_alt)
_alt_r2      = 1 - np.sum((_az_dev_alt_means - _alt_pred)**2) / np.sum((_az_dev_alt_means - _az_dev_alt_means.mean())**2)

print("\n── Component 2: Az-dependent offset (polar misalignment) ───")
print(f"   dev_roll sinusoid amplitude : {roll_az_amplitude:+.1f} arcmin  R²={_az_r2:.4f}")
print(f"   dev_roll Az of maximum      : {roll_az_az0:.1f}°")
print(f"   dev_alt  sinusoid amplitude : {_popt_alt[0]:+.1f} arcmin  R²={_alt_r2:.4f}  (corroboration)")
print(f"   dev_alt  Az of maximum      : {_popt_alt[1]:.1f}°")
print(f"   Implied polar tilt          : ~{abs(_popt_alt[0])/60:.2f}° toward Az≈{_popt_alt[1]:.0f}°")
print(f"   Interpretation              : QUEST absorbs this with multi-point sync spread across Az")

model_df['dev_roll_az_component'] = sinusoidal_az(model_df['p_az'], *_popt_az)
model_df['dev_roll_residual']     = model_df['dev_roll_arcmin'] - model_df['dev_roll_az_component']

# ── Component 3: Roll-dependent residual ──────────────────────────────────────

_cell_fits = []
for (az_bin, alt_bin), grp in model_df.groupby(['az_bin', 'alt_bin'], observed=True):
    if len(grp) < 10:
        continue
    slope, intercept, r, _, _ = stats.linregress(grp['p_roll'], grp['dev_roll_residual'])
    _cell_fits.append({
        'az_bin': az_bin, 'alt_bin': alt_bin,
        'mean_alt': grp['p_alt'].mean(),
        'slope': slope, 'intercept': intercept,
        'r2': r**2, 'n': len(grp),
    })
cell_fits = pd.DataFrame(_cell_fits)

print("\n── Component 3: Per-cell slope (dev_roll_residual vs p_roll) ")
print(f"   {'Az':>6}  {'Alt':>5}  {'slope':>7}  {'R²':>6}  {'n':>4}  quality")
for _, row in cell_fits.iterrows():
    quality = '✓ good' if row['r2'] >= 0.95 else ('~ ok' if row['r2'] >= 0.85 else '✗ weak')
    print(f"   {row['az_bin']:>6}  {row['alt_bin']:>5}  {row['slope']:>7.3f}  "
          f"{row['r2']:>6.3f}  {row['n']:>4.0f}  {quality}")

mean_r2_cells = cell_fits['r2'].mean()
print(f"\n   Mean per-cell R² : {mean_r2_cells:.4f}  (across {len(cell_fits)} cells)")

# Aggregate slope by alt bin (mean over Az panels)
_slope_by_alt = (
    cell_fits
    .groupby('alt_bin', observed=True)
    .agg(mean_alt=('mean_alt', 'mean'), mean_slope=('slope', 'mean'), slope_std=('slope', 'std'))
    .reindex(MODEL_ALT_BINS)
    .reset_index()
)

_tan_alts = np.tan(np.radians(_slope_by_alt['mean_alt'].values))
_slopes   = _slope_by_alt['mean_slope'].values

_s, _i, _r, _, _se = stats.linregress(_tan_alts, _slopes)
roll_model_a = _s
roll_model_b = _i

print("\n── Component 3: slope(alt) = roll_model_a · tan(alt) + roll_model_b")
print(f"   R²(slope vs tan(alt)) = {_r**2:.4f}")
print()
print(f"   {'Alt':>6}  {'tan(alt)':>9}  {'slope (actual)':>15}  {'±std Az':>8}  "
      f"{'slope (model)':>14}  {'residual':>9}  {'% err':>7}")
for _, row in _slope_by_alt.iterrows():
    tan_a   = np.tan(np.radians(row['mean_alt']))
    pred    = _s * tan_a + _i
    resid   = row['mean_slope'] - pred
    pct     = 100 * abs(resid) / row['mean_slope'] if row['mean_slope'] != 0 else float('nan')
    print(f"   {row['mean_alt']:>5.1f}°  {tan_a:>9.3f}  {row['mean_slope']:>15.3f}  "
          f"{row['slope_std']:>8.3f}  {pred:>14.3f}  {resid:>9.3f}  {pct:>6.1f}%")

# ── Model definition ──────────────────────────────────────────────────────────

def roll_error_model(p_roll_deg, p_alt_deg,
                     a=roll_model_a, b=roll_model_b):
    """
    Predicted roll error (arcmin) due to roll-dependent IMU bias.
    Subtract from IMU-reported roll to precondition before QUEST.

        corrected_roll_deg = p_roll_deg - roll_error_model(p_roll_deg, p_alt_deg) / 60
    """
    return (a * np.tan(np.radians(p_alt_deg)) + b) * p_roll_deg

# ── Validation: residuals before and after correction ─────────────────────────

model_df['roll_error_predicted'] = roll_error_model(model_df['p_roll'], model_df['p_alt'])
model_df['dev_roll_final']       = model_df['dev_roll_residual'] - model_df['roll_error_predicted']

_before_std  = model_df['dev_roll_residual'].std()
_after_std   = model_df['dev_roll_final'].std()
_improvement = 100 * (1 - _after_std / _before_std)

# Per-alt-bin validation R²
print("\n── Validation: residual after applying roll_error_model ────")
print(f"   {'Alt':>5}  {'before std':>11}  {'after std':>10}  {'reduction':>10}  {'R²(corrected)':>14}")
for alt_bin in MODEL_ALT_BINS:
    grp = model_df[model_df['alt_bin'] == alt_bin]
    b_std = grp['dev_roll_residual'].std()
    a_std = grp['dev_roll_final'].std()
    red   = 100 * (1 - a_std / b_std)
    # R² of model predictions vs residuals within this bin
    ss_res = np.sum(grp['dev_roll_final']**2)
    ss_tot = np.sum((grp['dev_roll_residual'] - grp['dev_roll_residual'].mean())**2)
    r2_bin = 1 - ss_res / ss_tot
    print(f"   {alt_bin:>5}  {b_std:>10.1f}′  {a_std:>9.1f}′  {red:>9.1f}%  {r2_bin:>14.4f}")

print(f"\n   Overall  {_before_std:>10.1f}′  {_after_std:>9.1f}′  {_improvement:>9.1f}%")

# ── Summary ───────────────────────────────────────────────────────────────────

print("\n── Model summary ────────────────────────────────────────────")
print(f"""
   Three-component decomposition of IMU pointing error:

   [1] Global SPA bias     : {_global_mean/60:+.3f}° roll offset (constant everywhere)
   [2] Polar misalignment  : {abs(_popt_alt[0])/60:.3f}° tilt toward Az≈{_popt_alt[1]:.0f}°  (sinusoidal in Az)
   [3] Roll-dependent bias : removed by preconditioning IMU quaternion  ← new

   Correction equation (Component 3):

       roll_error (arcmin) = (roll_model_a · tan(alt) + roll_model_b) · p_roll

   where:
       roll_model_a = {roll_model_a:.4f}   [arcmin per degree of p_roll, per unit tan(alt)]
       roll_model_b = {roll_model_b:.4f}   [arcmin per degree of p_roll, at horizon]
       p_roll       = IMU-reported roll angle (degrees)
       alt          = IMU-reported altitude (degrees)

   To apply:
       corrected_roll = p_roll  −  roll_error_model(p_roll, alt) / 60

   Components [1] and [2] are handled by QUEST alignment.
   Component  [3] must be corrected BEFORE passing the quaternion to QUEST.
""")

print("── Fitted coefficients ──────────────────────────────────────")
print(f"   roll_az_amplitude = {roll_az_amplitude:.4f}   # arcmin")
print(f"   roll_az_az0       = {roll_az_az0:.4f}   # degrees")
print(f"   roll_az_offset    = {roll_az_offset:.4f}   # arcmin  (= Component 1)")
print(f"   roll_model_a      = {roll_model_a:.4f}   # tan(alt) coefficient")
print(f"   roll_model_b      = {roll_model_b:.4f}   # horizon offset")

# ROLL_MODEL_A (0.9742) — this is the geometric projection coefficient. It describes how much a roll error in the camera frame projects onto the Az coordinate as altitude increases. The fact that it's close to 1.0 (not exactly 1.0) means the IMU's theta3 encoder has a small gain error — it reports slightly less rotation than actually occurred. This is a hardware characteristic of the specific Polaris unit (motor encoder linearity, flex in the M3 arm). It should be stable across setups and SPA alignments.
# ROLL_MODEL_B (0.2509) — this is the residual roll coupling at zero altitude, i.e. when tan(alt)=0 the model still predicts a small error proportional to p_roll. This represents a mechanical zero-point offset in the M3 encoder — the IMU thinks theta3=0 but the camera up-vector isn't quite where it thinks it is. This is also a hardware characteristic.

── Component 1: Global alignment offset (SPA bias) ─────────
   Mean dev_roll        : +147.2 arcmin  (+2.454°)
   Std  dev_roll        : 94.3 arcmin
   N (solved, 4×4 grid) : 878
   Interpretation       : mount has ~2.45° global roll bias under SPA-only
                          QUEST will absorb this with any sync point

── Component 2: Az-dependent offset (polar misalignment) ───
   dev_roll sinusoid amplitude : -78.9 arcmin  R²=0.9977
   dev_roll Az of maximum      : 104.1°
   dev_alt  sinusoid amplitude : +53.1 arcmin  R²=0.9994  (corroboration)
   dev_alt  Az of maximum      : 11.4°
   Implied polar tilt          : ~0.88° toward Az≈11°
   Interpretation              : QUEST absorbs this with multi-point sync spread across Az

── Component 3: Per-cell slope (dev_roll_residual vs p_roll) 
       Az    Alt    slope      R²     n  quality
     ~70°   ~25°    0.703   0.976    45  ✓ good
     ~70°   ~40°    0.950   0.960    67  ✓ good
     ~70°   ~55°    1.464   0.967    68  ✓ good
   

In [81]:
solved['roll_error_predicted'] = (0.9742 * np.tan(np.radians(solved['p_alt'])) + 0.2509) * solved['p_roll']

# Component 3 removed — should show no p_roll dependence, but still has C1+C2
solved['dev_roll_fixed'] = solved['dev_roll_arcmin'] - solved['roll_error_predicted']

# Also remove Az-dependent mean (C2) to isolate residual noise floor
solved['dev_roll_az_component'] = sinusoidal_az(solved['p_az'], *_popt_az)
solved['dev_roll_residual_final'] = solved['dev_roll_fixed'] - solved['dev_roll_az_component']

# Quick check — correlation with p_roll should drop toward zero
print("Correlation dev_roll vs p_roll         (before):", 
      solved['dev_roll_arcmin'].corr(solved['p_roll']).round(3))
print("Correlation dev_roll_fixed vs p_roll   (after C3):", 
      solved['dev_roll_fixed'].corr(solved['p_roll']).round(3))
print("Correlation dev_roll_residual vs p_roll (after C2+C3):", 
      solved['dev_roll_residual_final'].corr(solved['p_roll']).round(3))

# And the std should drop
print(f"\nStd before : {solved['dev_roll_arcmin'].std():.1f} arcmin")
print(f"Std after  : {solved['dev_roll_residual_final'].std():.1f} arcmin")

Correlation dev_roll vs p_roll         (before): 0.611
Correlation dev_roll_fixed vs p_roll   (after C3): -0.077
Correlation dev_roll_residual vs p_roll (after C2+C3): -0.052

Std before : 93.6 arcmin
Std after  : 33.0 arcmin


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ROTATION BIAS CORRECTION MODEL FIT
# Decomposes dev_roll into three components:
#   1. Global alignment offset (SPA-only bias — absorbed by QUEST in production)
#   2. Az-dependent offset     (polar misalignment — also absorbed by QUEST)
#   3. Roll-dependent residual (what QUEST cannot fix — the target of this model)
#
# Final model:
#   roll_error(arcmin) = (roll_model_a · tan(alt) + roll_model_b) · p_roll
# ─────────────────────────────────────────────────────────────────────────────

from scipy.optimize import curve_fit, minimize
from scipy import stats

# Work from all solved rows — no binning required
model_df = solved.copy()

# ── Component 1: Global SPA bias ──────────────────────────────────────────────

_global_mean = model_df['dev_roll_arcmin'].mean()
_global_std  = model_df['dev_roll_arcmin'].std()
_global_n    = len(model_df)

print("── Component 1: Global alignment offset (SPA bias) ─────────")
print(f"   Mean dev_roll        : {_global_mean:+.1f} arcmin  ({_global_mean/60:+.3f}°)")
print(f"   Std  dev_roll        : {_global_std:.1f} arcmin")
print(f"   N (solved)           : {_global_n}")
print(f"   Interpretation       : mount has ~{_global_mean/60:.2f}° global roll bias under SPA-only")
print(f"                          QUEST will absorb this with any sync point")

# ── Component 2: Az-dependent offset (polar misalignment) ─────────────────────
#
# Fit a sinusoid directly to the continuous (p_az, dev_roll) data via least squares,
# no binning needed — curve_fit works on the full point cloud.

def sinusoidal_az(az_deg, amplitude, az0_deg, offset):
    return amplitude * np.cos(np.radians(az_deg - az0_deg)) + offset

# Initial guess: amplitude from data range, az0=90, offset=mean
_p0_roll = [
    (model_df['dev_roll_arcmin'].max() - model_df['dev_roll_arcmin'].min()) / 2,
    90.0,
    model_df['dev_roll_arcmin'].mean()
]
_popt_az, _ = curve_fit(
    sinusoidal_az,
    model_df['p_az'].values,
    model_df['dev_roll_arcmin'].values,
    p0=_p0_roll,
    maxfev=10000
)
roll_az_amplitude, roll_az_az0, roll_az_offset = _popt_az

_az_pred_all = sinusoidal_az(model_df['p_az'].values, *_popt_az)
_az_r2 = 1 - np.sum((model_df['dev_roll_arcmin'].values - _az_pred_all)**2) / \
              np.sum((model_df['dev_roll_arcmin'].values - model_df['dev_roll_arcmin'].mean())**2)

# Independent confirmation from dev_alt sinusoid
_p0_alt = [
    (model_df['dev_alt_arcmin'].max() - model_df['dev_alt_arcmin'].min()) / 2,
    10.0,
    model_df['dev_alt_arcmin'].mean()
]
_popt_alt, _ = curve_fit(
    sinusoidal_az,
    model_df['p_az'].values,
    model_df['dev_alt_arcmin'].values,
    p0=_p0_alt,
    maxfev=10000
)
_alt_pred_all = sinusoidal_az(model_df['p_az'].values, *_popt_alt)
_alt_r2 = 1 - np.sum((model_df['dev_alt_arcmin'].values - _alt_pred_all)**2) / \
              np.sum((model_df['dev_alt_arcmin'].values - model_df['dev_alt_arcmin'].mean())**2)

print("\n── Component 2: Az-dependent offset (polar misalignment) ───")
print(f"   dev_roll sinusoid amplitude : {roll_az_amplitude:+.1f} arcmin  R²={_az_r2:.4f}")
print(f"   dev_roll Az of maximum      : {roll_az_az0:.1f}°")
print(f"   dev_alt  sinusoid amplitude : {_popt_alt[0]:+.1f} arcmin  R²={_alt_r2:.4f}  (corroboration)")
print(f"   dev_alt  Az of maximum      : {_popt_alt[1]:.1f}°")
print(f"   Implied polar tilt          : ~{abs(_popt_alt[0])/60:.2f}° toward Az≈{_popt_alt[1]:.0f}°")
print(f"   Interpretation              : QUEST absorbs this with multi-point sync spread across Az")

model_df['dev_roll_az_component'] = sinusoidal_az(model_df['p_az'], *_popt_az)
model_df['dev_roll_residual']     = model_df['dev_roll_arcmin'] - model_df['dev_roll_az_component']

# ── Component 3: Roll-dependent residual ──────────────────────────────────────
#
# Fit the full model in one step directly on the continuous data:
#   dev_roll_residual = (a · tan(alt) + b) · p_roll
#
# This is a linear regression with two engineered features:
#   X1 = tan(alt) · p_roll
#   X2 = p_roll

model_df['tan_alt']      = np.tan(np.radians(model_df['p_alt']))
model_df['feature_tanp'] = model_df['tan_alt'] * model_df['p_roll']   # X1
model_df['feature_p']    = model_df['p_roll']                          # X2

X = np.column_stack([
    model_df['feature_tanp'].values,
    model_df['feature_p'].values,
])
y = model_df['dev_roll_residual'].values

# Fit without intercept — the model is (a·tan(alt) + b)·p_roll,
# which is zero when p_roll=0 by construction (no free constant term)
from numpy.linalg import lstsq
coeffs, _, _, _ = lstsq(X, y, rcond=None)
roll_model_a, roll_model_b = coeffs

_pred_c3  = X @ coeffs
_ss_res   = np.sum((y - _pred_c3)**2)
_ss_tot   = np.sum((y - y.mean())**2)
_r2_c3    = 1 - _ss_res / _ss_tot
_rmse_c3  = np.sqrt(_ss_res / len(y))

print("\n── Component 3: Roll-dependent residual model ──────────────")
print(f"   Fit on {len(model_df)} points (continuous, no binning)")
print(f"   R²   = {_r2_c3:.4f}")
print(f"   RMSE = {_rmse_c3:.1f} arcmin")
print(f"\n   roll_fixed = p_roll - roll_error")
print(f"   roll_error = slope(alt) * p_roll")
print(f"   slope(alt) = roll_model_a · tan(alt) + roll_model_b")
print(f"              = {roll_model_a:.4f} · tan(alt) + {roll_model_b:.4f}")
print(f"\n   Implied slope at representative altitudes:")
print(f"   {'Alt':>6}  {'tan(alt)':>9}  {'slope':>7}")
for alt in [20, 30, 40, 50, 60, 70, 80]:
    s = roll_model_a * np.tan(np.radians(alt)) + roll_model_b
    print(f"   {alt:>5}°  {np.tan(np.radians(alt)):>9.3f}  {s:>7.3f}")

# ── Model definition ──────────────────────────────────────────────────────────

def roll_error_model(p_roll_deg, p_alt_deg,
                     a=roll_model_a, b=roll_model_b):
    """
    Predicted roll error (arcmin) due to roll-dependent IMU bias.
    Subtract from IMU-reported roll to precondition before QUEST.

        corrected_roll_deg = p_roll_deg - roll_error_model(p_roll_deg, p_alt_deg) / 60
    """
    return (a * np.tan(np.radians(p_alt_deg)) + b) * p_roll_deg

# ── Validation ────────────────────────────────────────────────────────────────

model_df['roll_error_predicted'] = roll_error_model(model_df['p_roll'], model_df['p_alt'])
model_df['dev_roll_final']       = model_df['dev_roll_residual'] - model_df['roll_error_predicted']

_before_std  = model_df['dev_roll_residual'].std()
_after_std   = model_df['dev_roll_final'].std()
_improvement = 100 * (1 - _after_std / _before_std)

_corr_before = model_df['dev_roll_residual'].corr(model_df['p_roll'])
_corr_after  = model_df['dev_roll_final'].corr(model_df['p_roll'])

print("\n── Validation ───────────────────────────────────────────────")
print(f"   Correlation with p_roll  before : {_corr_before:+.4f}")
print(f"   Correlation with p_roll  after  : {_corr_after:+.4f}")
print(f"   Std before : {_before_std:.1f} arcmin")
print(f"   Std after  : {_after_std:.1f} arcmin  ({_improvement:.1f}% reduction)")

# ── Summary ───────────────────────────────────────────────────────────────────

print("\n── Model summary ────────────────────────────────────────────")
print(f"""
   Three-component decomposition of IMU pointing error:

   [1] Global SPA bias     : {_global_mean/60:+.3f}° roll offset (constant everywhere)
   [2] Polar misalignment  : {abs(_popt_alt[0])/60:.3f}° tilt toward Az≈{_popt_alt[1]:.0f}°  (sinusoidal in Az)
   [3] Roll-dependent bias : removed by preconditioning IMU quaternion  ← new

   Correction equation (Component 3):

       roll_error (arcmin) = (roll_model_a · tan(alt) + roll_model_b) · p_roll

   where:
       roll_model_a = {roll_model_a:.4f}   [arcmin/° of p_roll per unit tan(alt)]
       roll_model_b = {roll_model_b:.4f}   [arcmin/° of p_roll at horizon]
       p_roll       = IMU-reported roll angle (degrees)
       alt          = IMU-reported altitude (degrees)

   To apply:
       corrected_roll = p_roll  −  roll_error_model(p_roll, alt) / 60

   Components [1] and [2] are handled by QUEST alignment.
   Component  [3] must be corrected BEFORE passing the quaternion to QUEST.
""")

print("── Fitted coefficients ──────────────────────────────────────")
print(f"   roll_az_amplitude = {roll_az_amplitude:.4f}   # arcmin (diagnostic only)")
print(f"   roll_az_az0       = {roll_az_az0:.4f}   # degrees (diagnostic only)")
print(f"   roll_az_offset    = {roll_az_offset:.4f}   # arcmin (diagnostic only)")
print(f"   roll_model_a      = {roll_model_a:.4f}   # ← use in driver")
print(f"   roll_model_b      = {roll_model_b:.4f}   # ← use in driver")

── Component 1: Global alignment offset (SPA bias) ─────────
   Mean dev_roll        : +147.2 arcmin  (+2.453°)
   Std  dev_roll        : 93.6 arcmin
   N (solved)           : 893
   Interpretation       : mount has ~2.45° global roll bias under SPA-only
                          QUEST will absorb this with any sync point

── Component 2: Az-dependent offset (polar misalignment) ───
   dev_roll sinusoid amplitude : -77.5 arcmin  R²=0.3385
   dev_roll Az of maximum      : 103.0°
   dev_alt  sinusoid amplitude : +53.3 arcmin  R²=0.6804  (corroboration)
   dev_alt  Az of maximum      : 10.7°
   Implied polar tilt          : ~0.89° toward Az≈11°
   Interpretation              : QUEST absorbs this with multi-point sync spread across Az

── Component 3: Roll-dependent residual model ──────────────
   Fit on 893 points (continuous, no binning)
   R²   = 0.8122
   RMSE = 33.0 arcmin

   roll_fixed = p_roll - roll_error
   roll_error = slope(alt) * p_roll
   slope(alt) = roll_model_a · tan(alt)